In [0]:
!pip install langgraph
!pip install langchain
!pip install pygame
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install -U langchain-openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# %pip uninstall -y langchain langchain-community langchain-core langchain-databricks
# %pip install -U databricks-langchain langgraph
# dbutils.library.restartPython()

In [0]:

# from databricks_langchain import ChatDatabricks

# llm = ChatDatabricks(endpoint="databricks-meta-llama-3-70b-instruct")


In [0]:
import os
import mlflow
import subprocess
import operator
from typing import TypedDict, Annotated, List
from langgraph.graph.message import add_messages
from typing import Callable
from dataclasses import dataclass
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


##Task 1: Initializing System Memory and State Management

In [0]:
import os
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = ""
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)

In [0]:
class GameState(TypedDict):
    ## TO-DO
    director_messages: Annotated[list, add_messages]
    architect_messages: Annotated[list, add_messages]
    engineer_code: Annotated[list, add_messages]
    qa_feedback: Annotated[list, add_messages]
    current_actor: str
    iteration: int
    iteration_score: Annotated[List[int], operator.add]
    file_saved: bool


### Subtask 2.1: Director and Architect Nodes

In [0]:
def get_clean_text(messages_list):
    if not messages_list:
        return ""
    latest = messages_list[-1]
    if hasattr(latest, 'content'):
        return str(latest.content)
    return str(latest)

def director_node(state: GameState):
    director_msg = input("Awaiting Director Prompt: ")
    return {"director_messages": [director_msg], "current_actor": "director"}

In [0]:
def director_node(state: GameState):
    director_msg = input("Awaiting Director Prompt: ")
    return {
        "director_messages": [director_msg],
        "current_actor": "director"
    }


def architect_node(state: GameState):
    print(f"\n========== ITERATION {state.get('iteration', 0)} ==========")
    
    latest_prompt = get_clean_text(state.get("director_messages", []))
    
    system_prompt = f"""You are an expert Systems Architect for Python game development.
    Based on the following request, output a clean, numbered high-level system design.
    Keep it concise.
    
    Director Request: {latest_prompt}"""
    
    print("Architect is generating design...")
    response = llm.invoke(system_prompt)
    architect_design = response.content
    
    return {"architect_messages": [architect_design], "current_actor": "architect"}

### Subtask 2.2: Engineer Node

In [0]:
def engineer_node(state: GameState):
    print("\nEngineer is writing code...")
    
    arch_msg = get_clean_text(state.get("architect_messages", []))
    qa_msg = get_clean_text(state.get("qa_feedback", []))
    system_prompt = f"""You are an expert Python Game Developer.
    Based on this architecture: {arch_msg}
    And this QA feedback: {qa_msg}
    
    Write a fully functional 2D endless runner game using the 'pygame' library. 
    Include flying obstacles (pterodactyls), ground obstacles (cacti), jumping, ducking, and a score tracker.
    
    CRITICAL: Output ONLY raw, executable Python code. Do not include markdown formatting like ```python or any explanations."""
    
    response = llm.invoke(system_prompt)
    new_code = response.content.strip()
    
    if new_code.startswith("```python"):
        new_code = new_code[9:]
    if new_code.endswith("```"):
        new_code = new_code[:-3]
    
    return {"engineer_code": [new_code], "current_actor": "engineer"}


### Subtask 2.3: File I/O and Execution Nodes

In [0]:
def file_writer(state: GameState):
    code = state.get("engineer_code", [])
    code_msg = code[-1] if code else ""
    if hasattr(code_msg, 'content'):
        code =code_msg.content
    else:
        code = str(code_msg)
    with open("dino_runner.py", "w", encoding="utf-8") as f:
        f.write(code_msg)

    print(code_msg[:500])
    print("\n[File saved: dino_runner.py]")
    return {
        "file_saved": True,
        "current_actor": "file_writer"
    }


def run_code(state: GameState):
    print("\n===== CODE EXECUTION =====")
    choice = input("Run the generated game? (y/n): ").strip().lower()
    execution_result = ""
    if choice == 'y':
        try:
            result = subprocess.run(
                ["python", "dino_runner.py"],
                capture_output=True,
                text=True,
                timeout=15
            )
            if result.returncode == 0:
                execution_result = f"Execution Successful.\nStandard Output:\n{result.stdout}"
            else:
                execution_result = f"Runtime Error.\nStandard Error:\n{result.stderr}"
        except Exception as e:
            execution_result = f"System Error during execution: {str(e)}"
    else:
        execution_result = "Execution aborted by user intervention."
        
    return {
        "qa_feedback": [execution_result],
        "current_actor": "execution_manager"
    }

### Subtask 2.4: QA and Scorer Nodes

In [0]:
def qa_node(state: GameState):
    out = get_clean_text(state.get("qa_feedback", []))
    design_requirements = get_clean_text(state.get("architect_messages", []))
    engineer_code = get_clean_text(state.get("engineer_code", []))
    
    # Now it is perfectly safe to slice [:50] and use len()
    qa_report = f"QA Report based on execution:\nOutput: {out[:100]}...\nDesign Alignment: Checked against {design_requirements[:50]}...\nCode Health: Evaluated {len(engineer_code)} characters."
    
    return {
        "qa_feedback": [qa_report],
        "current_actor": "qa_engineer"
    }

def score_node(state: GameState):
    qa_logs = state.get("qa_feedback", [])
    qa_feedback = qa_logs[-1] if qa_logs else ""
    
    score = 8 
    
    return {
        "iteration_score": [score],
        "current_actor": "scorer"
    }




In [0]:
def should_continue(state: GameState):
    scores = state.get("iteration_score", [])
    latest_score = scores[-1] if scores else 0
    print(f"System Score for this iteration: {latest_score}/10")
    choice = input("Trigger a new iteration to refine the code? (y/n): ").strip().lower()
    
    if choice == 'y':
        return "continue"
    else:
        return "end"

## Task 3: Create Graph Structure 


In [0]:

workflow = StateGraph(GameState)
workflow.add_node("director", director_node)
workflow.add_node("architect", architect_node)
workflow.add_node("engineer", engineer_node)
workflow.add_node("file_writer", file_writer)
workflow.add_node("run_code", run_code)
workflow.add_node("qa", qa_node)
workflow.add_node("scorer", score_node)
workflow.add_edge(START, "director")
workflow.add_edge("director", "architect")
workflow.add_edge("architect", "engineer")
workflow.add_edge("engineer", "file_writer")
workflow.add_edge("file_writer", "run_code")
workflow.add_edge("run_code", "qa")
workflow.add_edge("qa", "scorer")

workflow.add_conditional_edges(
    "scorer",
    should_continue,
    {
        "continue": "engineer",
        "end": END
    }
)

app = workflow.compile()

## Task 4: System Invocation

In [0]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "dino_runner_session_2"}}
print("--- STARTING WORKFLOW ---")
for event in app.stream({}, config):
    for node, value in event.items():
        print(f"\n===== {node.upper()} =====")
        print(value)


--- STARTING WORKFLOW ---


Awaiting Director Prompt:  Build a 2D endless runner game in the style of the Chrome Dino game. The player controls a character that must jump over ground obstacles like cacti and duck under flying obstacles like pterodactyls. The game needs accurate gravity and jumping physics, keyboard input mapping for jump and duck, and a functional score and high-score tracking system that increases in speed over time


===== DIRECTOR =====
{'director_messages': ['Build a 2D endless runner game in the style of the Chrome Dino game. The player controls a character that must jump over ground obstacles like cacti and duck under flying obstacles like pterodactyls. The game needs accurate gravity and jumping physics, keyboard input mapping for jump and duck, and a functional score and high-score tracking system that increases in speed over time'], 'current_actor': 'director'}

========== ITERATION 0 ==========
Architect is generating design...

===== ARCHITECT =====
{'architect_messages': ['1. **Game Core Loop**\n   - Fixed-time update loop for deterministic physics and collision handling.\n   - Separate `update()` and `render()` phases.\n   - Track game states: `menu`, `playing`, `game_over`, `restart`.\n\n2. **Player Movement System**\n   - Player entity with position, velocity, acceleration, and state flags: `running`, `jumping`, `ducking`.\n   - Jump action applies an initial upward velocity.\n   - Du

Run the generated game? (y/n):  n


===== RUN_CODE =====
{'qa_feedback': ['Execution aborted by user intervention.'], 'current_actor': 'execution_manager'}

===== QA =====
{'qa_feedback': ['QA Report based on execution:\nOutput: Execution aborted by user intervention....\nDesign Alignment: Checked against 1. **Game Core Loop**\n   - Fixed-time update loop ...\nCode Health: Evaluated 15595 characters.'], 'current_actor': 'qa_engineer'}
System Score for this iteration: 8/10


Trigger a new iteration to refine the code? (y/n):  n


===== SCORER =====
{'iteration_score': [8], 'current_actor': 'scorer'}
